# Teva Pharmaceutical Industries -- Pentest Recon (Colab runner)

Wraps `pentest-teva/tools/teva_pentest_all.py` from the HAMIVTZAR repo for a Colab (or any hosted-notebook) environment. Read this whole cell before running anything.

## Do not run the network-facing cells yet

Per `pentest-teva/scope.md`, these are still **unresolved**:

- [ ] Signed-agreement reference (contract/engagement number, signatory name and title, effective date)
- [ ] Engagement window (start/end dates)
- [ ] **Authorized tester source IP(s)** -- Colab's outbound IP is shared, ephemeral, and drawn from Google Cloud's pool. It cannot be "the pre-registered authorized source IP" the way a real engagement expects, without a separate arrangement (e.g. a static/elastic IP on a cloud VM, registered with Teva's security contact ahead of time). Running from plain Colab means this traffic is not attributable to the engagement from Teva's side.
- [ ] Emergency-stop / real-time security contact

`AUTHORIZED_DOMAINS` in the tool already contains `teva.co.il` and `tevapharm.com` (named by the requester) -- that only means the tool's own code-level domain gate won't block you. It does **not** mean the items above are resolved.

**Only the `selftest` and `cve` cells below are safe to run right now** (fully offline, no network). Every other cell checks a `SCOPE_CONFIRMED` flag and refuses to run until you deliberately set it to `True` -- do that only after the open items above are actually resolved and `scope.md` is updated to reflect it.

## 1. Setup

In [ ]:
# impacket/ldap3 are only needed for the `smb`/`ldap` subcommands (internal,
# anonymous enumeration). dns/paths/adpaths/ntlminfo/cve/selftest need only stdlib.
!pip install -q impacket ldap3


## 2. Write out the tool

Embedded verbatim from `pentest-teva/tools/teva_pentest_all.py` in the HAMIVTZAR repo (as of when this notebook was generated) -- self-contained on purpose, no other repo files needed.

In [ ]:
%%writefile teva_pentest_all.py
#!/usr/bin/env python3
"""Teva Pharmaceutical Industries engagement tool -- all-in-one.

Retargeted from a tool built for a different, unrelated engagement (a
municipal target) -- see ../scope.md for why that matters and what's still
unconfirmed here. This is the ONLY tool file for this engagement, meant to
be copyable to a network-permitted machine on its own (no other files from
this repo needed). Same guardrails throughout -- see ../scope.md (or ask
whoever gave you this file for a copy of it) before running anything.

Domains this file will run external checks against are an explicit
allowlist (AUTHORIZED_DOMAINS below): currently `teva.co.il` and
`tevapharm.com`, the two corporate domains named by the requester. Naming
the domains is NOT the same as confirming the rest of the engagement --
see ../scope.md's open items (engagement window, authorized source IPs,
emergency contact, signed-agreement reference) which are still unresolved.
Adding a further entry here is a deliberate code change gated on that
domain having its own signed-engagement record confirmed in ../scope.md
first -- not a runtime flag and never "any Teva-looking domain".

Subcommands:
    dns       [domain]                 passive DNS + multi-source certificate-transparency recon (external)
    siteinfo  <base-url>                fetch + parse robots.txt and sitemap.xml, only if they 200 (external)
    paths     <base-url>               curated disclosure-path check against one host (external)
    adpaths   [domain]                 DNS -> SUBDOMAINS -> PATHS: curated AD/SSO-pivot path check on discovered live subdomains
    ntlminfo  <url>                    passive NTLM auth-challenge domain-name disclosure against one URL (external)
    cve       --product P [--version V]  static, offline known-CVE reference lookup -- NO network access
    smb       <internal-host-or-ip>    anonymous SMB null-session enum (internal)
    ldap      <internal-host-or-ip>    anonymous LDAP bind enum (internal)
    selftest                           run this file's own embedded offline unit tests

Examples (replace <authorized-domain> with an entry you've actually added
to AUTHORIZED_DOMAINS after confirming it in ../scope.md):
    python3 teva_pentest_all.py dns
    python3 teva_pentest_all.py dns <authorized-domain>
    python3 teva_pentest_all.py siteinfo https://www.<authorized-domain>
    python3 teva_pentest_all.py paths https://www.<authorized-domain>
    python3 teva_pentest_all.py adpaths <authorized-domain>
    python3 teva_pentest_all.py ntlminfo https://<authorized-domain>/some/observed/401/url
    python3 teva_pentest_all.py cve --product fortinet --version 6.4.6
    python3 teva_pentest_all.py smb 10.20.30.40
    python3 teva_pentest_all.py ldap 10.20.30.10
    python3 teva_pentest_all.py selftest

Do NOT run with sudo -- nothing here needs root, and sudo's Python often
can't see packages installed with `pip install --user`, which just produces
confusing "not installed" errors for packages that ARE installed.

This file is self-contained on purpose (see above): its own regression
tests are embedded below (see `selftest`) and run with only the stdlib, so
verifying it doesn't require the rest of this repo either.
"""

import argparse
import base64
import http.client
import json
import os
import socket
import ssl
import struct
import sys
import time
import unittest
import urllib.error
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from pathlib import Path

# ---------------------------------------------------------------------------
# Shared config / guardrails
# ---------------------------------------------------------------------------

# Domains this tool is authorized to run checks against. Adding an entry
# here is a deliberate code change that must only happen once a signed
# engagement authorizes that target -- see ../scope.md for the model
# (target, authorization, dates, rules of engagement recorded before any
# tooling runs against it). This is NOT a runtime flag and NOT "any
# Teva-looking domain": each entry needs its own such record.
#
# This scaffold was retargeted from a different, unrelated engagement (see
# ../scope.md). The two entries below are the corporate domains the
# requester named -- that is NOT the same as the rest of the engagement
# being confirmed: ../scope.md's other open items (engagement window,
# authorized source IPs, emergency contact, signed-agreement reference)
# are still unresolved, and this scaffold's own author has not
# independently verified any of it. Every external subcommand below still
# only runs against an exact match/subdomain of one of these two entries.
AUTHORIZED_DOMAINS = [
    "teva.co.il",     # see ../scope.md -- named by requester; other open items still unresolved
    "tevapharm.com",  # see ../scope.md -- named by requester; other open items still unresolved
]

# Specific hosts confirmed OUT of scope even though they're subdomains of an
# AUTHORIZED_DOMAINS entry -- per scope.md's "Out of scope" section (e.g.
# third-party-hosted infrastructure that merely happens to live under the
# domain in DNS). Checked before AUTHORIZED_DOMAINS matching, so an excluded
# host is refused even though it would otherwise match. Starts empty --
# nothing has been discovered/excluded yet for this engagement. Add here
# only once scope.md's Out-of-scope section records why.
EXCLUDED_HOSTS = [
]

USER_AGENT = "Mozilla/5.0 (compatible; authorized-pentest-recon/1.0; scope=see scope.md)"
DEFAULT_DELAY_SECONDS = 1.0
DEFAULT_TIMEOUT_SECONDS = 15

SCOPE_BANNER = """\
==============================================================================
 Teva Pharmaceutical Industries engagement tool -- read this before running
 against a live target, internal or external.
 - This scaffold was retargeted from a different, unrelated engagement.
   Confirm scope.md reflects a currently-valid, Teva-specific signed
   engagement -- not an inherited one -- and that every open item there is
   resolved before running anything beyond `selftest`.
 - Read-only. No auth bypass, brute-force, credential use, writes, or any
   AD attack technique (relay/poisoning/roasting) -- see scope.md.
 - Blackbox: no internal IP/hostname is hardcoded anywhere in this script.
   You must supply any internal target explicitly, and only after the
   engagement has actually granted you that network vantage point.
 - Requests are rate-limited; do not remove the delay to "go faster".
 - Do not run this with sudo -- it doesn't need root, and sudo often can't
   see `pip install --user` packages, which just causes confusing errors.
==============================================================================
"""

# Evidence lands next to this script (standalone-file layout), not two
# directories up -- unlike the multi-file repo layout this was split from.
EVIDENCE_DIR = Path(__file__).resolve().parent / "evidence"


def print_banner():
    print(SCOPE_BANNER, file=sys.stderr)


def save_evidence(name, data):
    EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    path = EVIDENCE_DIR / f"{stamp}_{name}.json"
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False))
    print(f"[evidence] saved {path}", file=sys.stderr)
    return path


def polite_get(url, delay=DEFAULT_DELAY_SECONDS, timeout=DEFAULT_TIMEOUT_SECONDS, insecure=False, extra_headers=None):
    """GET a URL with a fixed User-Agent, then sleep. Never raises -- always
    returns a dict describing the outcome, including connection-level
    failures (reset, dropped without a response, TLS error, timeout), which
    urllib does NOT wrap as urllib.error.URLError when raised from
    getresponse() -- they must be caught separately or a single flaky/WAF-
    reset request kills the whole run.

    insecure=True skips TLS certificate verification (like `curl -k`) --
    only affects whether *this client* trusts the cert; it doesn't change
    anything about the target. Useful when the local CA store can't
    validate the chain (missing intermediate, incomplete ca-certificates
    package) but you still want to see what the server actually serves for
    passive fingerprinting purposes.

    extra_headers merges into the request (e.g. ntlminfo's Authorization:
    NTLM header) -- still a single unauthenticated-by-default GET, never a
    real credential."""
    headers = {"User-Agent": USER_AGENT}
    if extra_headers:
        headers.update(extra_headers)
    req = urllib.request.Request(url, headers=headers)
    result = {"url": url, "requested_at": datetime.now(timezone.utc).isoformat()}
    ssl_context = None
    if insecure and url.lower().startswith("https://"):
        ssl_context = ssl.create_default_context()
        ssl_context.check_hostname = False
        ssl_context.verify_mode = ssl.CERT_NONE
    try:
        open_kwargs = {"timeout": timeout}
        if ssl_context is not None:
            open_kwargs["context"] = ssl_context
        with urllib.request.urlopen(req, **open_kwargs) as resp:
            body = resp.read(1_000_000)  # cap: don't pull huge bodies into memory
            result["status"] = resp.status
            result["headers"] = dict(resp.getheaders())
            result["body"] = body.decode("utf-8", errors="replace")
    except urllib.error.HTTPError as e:
        body = e.read(1_000_000) if e.fp else b""
        result["status"] = e.code
        result["headers"] = dict(e.headers.items()) if e.headers else {}
        result["body"] = body.decode("utf-8", errors="replace")
    except urllib.error.URLError as e:
        result["status"] = None
        result["error"] = str(e.reason)
        result["headers"] = {}
        result["body"] = ""
    except (OSError, http.client.HTTPException, ssl.SSLError, socket.timeout) as e:
        result["status"] = None
        result["error"] = f"{type(e).__name__}: {e}"
        result["headers"] = {}
        result["body"] = ""
    finally:
        time.sleep(delay)
    return result


def is_excluded_host(hostname):
    hostname = (hostname or "").lower()
    return any(hostname == h or hostname.endswith("." + h) for h in EXCLUDED_HOSTS)


def is_authorized_domain(hostname):
    hostname = (hostname or "").lower()
    if is_excluded_host(hostname):
        return False
    return any(hostname == d or hostname.endswith("." + d) for d in AUTHORIZED_DOMAINS)


def require_authorized_domain(domain_or_url, label="target"):
    """Extract a hostname from a bare domain or a full URL and refuse to
    continue unless it matches (or is a subdomain of) an entry in
    AUTHORIZED_DOMAINS and isn't in EXCLUDED_HOSTS. This is an exact suffix
    match, not a loose substring check -- a substring check would wrongly
    accept a look-alike host like "teva.co.il.evil.example" just because
    it contains the right text."""
    candidate = domain_or_url.strip()
    parsed = urllib.parse.urlparse(candidate if "://" in candidate else f"//{candidate}")
    hostname = parsed.hostname or ""
    if is_excluded_host(hostname):
        print(
            f"refusing to run: {label} {domain_or_url!r} (host {hostname!r}) is "
            "explicitly EXCLUDED from scope (see EXCLUDED_HOSTS and ../scope.md's "
            "Out-of-scope section) even though it's a subdomain of an authorized "
            "domain.",
            file=sys.stderr,
        )
        sys.exit(2)
    if not is_authorized_domain(hostname):
        print(
            f"refusing to run: {label} {domain_or_url!r} (host {hostname!r}) is "
            f"not in AUTHORIZED_DOMAINS ({', '.join(AUTHORIZED_DOMAINS)}) -- this "
            "tool only runs against domains with their own signed-engagement "
            "record (see ../scope.md); add a domain to AUTHORIZED_DOMAINS only "
            "after that record exists for it.",
            file=sys.stderr,
        )
        sys.exit(2)
    return domain_or_url


SUSPICIOUS_MIN_BYTES = 10


def looks_like_hit(result):
    """Shared by `paths` and `adpaths` below: a non-404-looking response
    worth a human's attention, not a confirmed finding."""
    status = result.get("status")
    body = result.get("body", "")
    if status == 200 and len(body) >= SUSPICIOUS_MIN_BYTES:
        return True
    if status in (401, 403):
        return True
    return False


# ---------------------------------------------------------------------------
# dns -- passive DNS/OSINT recon (external, no dependencies)
# ---------------------------------------------------------------------------

CANDIDATE_SUBDOMAINS = [
    "www", "mail", "webmail", "autodiscover", "vpn", "remote", "owa",
    "portal", "ns1", "ns2", "mx", "ftp",
]

def crtsh_url(domain):
    return f"https://crt.sh/?q=%25.{domain}&output=json"


def dns_resolve(hostname):
    try:
        infos = socket.getaddrinfo(hostname, None)
        addrs = sorted({info[4][0] for info in infos})
        return {"hostname": hostname, "addresses": addrs, "resolved": True}
    except socket.gaierror as e:
        return {"hostname": hostname, "error": str(e), "resolved": False}


def query_crtsh(domain):
    req = urllib.request.Request(
        crtsh_url(domain), headers={"User-Agent": "Mozilla/5.0 (compatible; authorized-pentest-recon/1.0)"}
    )
    try:
        with urllib.request.urlopen(req, timeout=20) as resp:
            data = json.loads(resp.read().decode("utf-8", errors="replace"))
            names = sorted({row.get("name_value", "").strip().lower() for row in data})
            return {"ok": True, "names": [n for n in names if n]}
    except (urllib.error.URLError, urllib.error.HTTPError, TimeoutError, json.JSONDecodeError,
            OSError, http.client.HTTPException, ssl.SSLError) as e:
        return {"ok": False, "error": f"{type(e).__name__}: {e}"}


def query_certspotter(domain):
    """SSLMate's Cert Spotter API (api.certspotter.com) -- free, no-auth
    certificate-transparency search, independent of crt.sh's own
    infrastructure. A second CT source matters because crt.sh is a single
    service run by one operator and has had outages/rate-limit blocks in
    the past -- this doesn't depend on the same infrastructure staying up.
    Third-party service, not the target: this queries SSLMate, never sends
    anything to the domain itself."""
    url = (
        "https://api.certspotter.com/v1/issuances"
        f"?domain={urllib.parse.quote(domain)}&include_subdomains=true&expand=dns_names"
    )
    req = urllib.request.Request(
        url, headers={"User-Agent": "Mozilla/5.0 (compatible; authorized-pentest-recon/1.0)"}
    )
    try:
        with urllib.request.urlopen(req, timeout=20) as resp:
            data = json.loads(resp.read().decode("utf-8", errors="replace"))
            names = set()
            for issuance in data:
                for dns_name in issuance.get("dns_names") or []:
                    dns_name = dns_name.strip().lower()
                    if dns_name:
                        names.add(dns_name)
            return {"ok": True, "names": sorted(names)}
    except (urllib.error.URLError, urllib.error.HTTPError, TimeoutError, json.JSONDecodeError,
            OSError, http.client.HTTPException, ssl.SSLError) as e:
        return {"ok": False, "error": f"{type(e).__name__}: {e}"}


def query_censys(domain):
    """Censys Search API -- optional, credentialed passive-recon source.
    Only attempted if CENSYS_API_ID / CENSYS_API_SECRET are set in the
    environment (your own account, your own quota) -- never hardcoded,
    never required, and an unset credential is a skip, not an error. Uses
    HTTP Basic auth (API ID as username, secret as password) against the
    certificates search endpoint -- verify against Censys's current docs if
    this stops matching their API shape; like every source here, a failure
    just reports {"ok": False} rather than breaking dns/adpaths."""
    api_id = os.environ.get("CENSYS_API_ID")
    api_secret = os.environ.get("CENSYS_API_SECRET")
    if not api_id or not api_secret:
        return {"ok": False, "skipped": True, "error": "CENSYS_API_ID/CENSYS_API_SECRET not set in environment"}

    query = urllib.parse.urlencode({"q": f"names: {domain}", "per_page": 100})
    url = f"https://search.censys.io/api/v2/certificates/search?{query}"
    credentials = base64.b64encode(f"{api_id}:{api_secret}".encode("ascii")).decode("ascii")
    req = urllib.request.Request(
        url,
        headers={
            "User-Agent": "Mozilla/5.0 (compatible; authorized-pentest-recon/1.0)",
            "Authorization": f"Basic {credentials}",
        },
    )
    try:
        with urllib.request.urlopen(req, timeout=20) as resp:
            data = json.loads(resp.read().decode("utf-8", errors="replace"))
            names = set()
            for hit in data.get("result", {}).get("hits", []) or []:
                for name in hit.get("names") or []:
                    name = name.strip().lower()
                    if name:
                        names.add(name)
            return {"ok": True, "names": sorted(names)}
    except (urllib.error.URLError, urllib.error.HTTPError, TimeoutError, json.JSONDecodeError,
            OSError, http.client.HTTPException, ssl.SSLError) as e:
        return {"ok": False, "error": f"{type(e).__name__}: {e}"}


def discover_ct_names(domain):
    """Query every configured certificate-transparency / passive-recon
    source for names under `domain` and merge into one sorted,
    deduplicated list. Each source is independent and best-effort: crt.sh
    going down, changing its response format, or rate-limiting doesn't lose
    Cert Spotter's results, and vice versa. Per-source raw results are kept
    too (not just the merge) so evidence/ shows exactly which service found
    what -- useful for a real report, and for noticing if one source
    silently stops working. All three are third-party services, not the
    target -- none of this sends anything to `domain` itself."""
    sources = {
        "crtsh": query_crtsh(domain),
        "certspotter": query_certspotter(domain),
        "censys": query_censys(domain),
    }
    merged = set()
    for source_result in sources.values():
        if source_result.get("ok"):
            merged.update(source_result.get("names", []))
    return {"sources": sources, "names": sorted(merged)}


def cmd_dns(args):
    print_banner()
    domain = args.domain
    if domain is None:
        if not AUTHORIZED_DOMAINS:
            print(
                "usage: dns <domain> -- AUTHORIZED_DOMAINS is empty, so there is "
                "nothing to default to. Resolve ../scope.md's open items and add a "
                "domain there first.",
                file=sys.stderr,
            )
            sys.exit(2)
        if len(AUTHORIZED_DOMAINS) != 1:
            print(
                "usage: dns <domain> -- multiple AUTHORIZED_DOMAINS are configured, "
                f"pick one explicitly: {', '.join(AUTHORIZED_DOMAINS)}",
                file=sys.stderr,
            )
            sys.exit(2)
        domain = AUTHORIZED_DOMAINS[0]
    require_authorized_domain(domain, label="domain")

    print(f"[*] Passive DNS recon for {domain}\n", file=sys.stderr)

    results = {"base_domain": dns_resolve(domain), "candidates": []}
    print(f"  {domain:30s} -> {results['base_domain']}")

    for sub in CANDIDATE_SUBDOMAINS:
        hostname = f"{sub}.{domain}"
        if is_excluded_host(hostname):
            print(f"  {hostname:30s} -> EXCLUDED (out of scope, see scope.md -- not queried)")
            results["candidates"].append({"hostname": hostname, "excluded": True})
            continue
        r = dns_resolve(hostname)
        results["candidates"].append(r)
        status = ", ".join(r["addresses"]) if r["resolved"] else "NXDOMAIN/error"
        print(f"  {hostname:30s} -> {status}")

    print(
        "\n[*] Querying certificate-transparency / passive-recon sources "
        "(third-party, not the target: crt.sh, Cert Spotter, optionally "
        "Censys)...",
        file=sys.stderr,
    )
    ct = discover_ct_names(domain)
    results["ct_sources"] = ct
    for source_name, source_result in ct["sources"].items():
        if source_result.get("ok"):
            print(f"  {source_name}: {len(source_result['names'])} name(s)")
        elif source_result.get("skipped"):
            print(f"  {source_name}: skipped -- {source_result['error']}")
        else:
            print(f"  {source_name}: failed -- {source_result.get('error')}", file=sys.stderr)
    if ct["names"]:
        print(f"\n  {len(ct['names'])} distinct name(s) across all sources:")
        for n in ct["names"]:
            print(f"    {n}")

    save_evidence("dns_recon", results)


# ---------------------------------------------------------------------------
# paths -- curated disclosure-path check (external, no dependencies)
# ---------------------------------------------------------------------------

# Generic web-hygiene/disclosure paths -- deliberately distinct from
# AD_PIVOT_PATHS below (that list hunts for AD/SSO integration points; this
# one is the standard "did something get left exposed" sweep: VCS/env
# files, backups, admin panels, API introspection endpoints). Curated,
# fixed list -- not a wordlist-driven brute force. Add to this deliberately
# and update scope.md first.
CANDIDATE_PATHS = [
    "/.git/HEAD", "/.git/config", "/.env", "/.env.local", "/.DS_Store",
    "/.htaccess", "/web.config", "/backup.zip", "/backup.sql", "/phpinfo.php",
    "/server-status", "/admin/", "/administrator/", "/swagger.json",
    "/swagger-ui/", "/graphql", "/robots.txt", "/sitemap.xml",
    "/security.txt", "/.well-known/security.txt",
]


def cmd_paths(args):
    print_banner()
    base = require_authorized_domain(args.base_url.rstrip("/"), label="target")

    print(f"[*] Checking {len(CANDIDATE_PATHS)} curated paths against {base}\n", file=sys.stderr)
    hits = []
    for path in CANDIDATE_PATHS:
        url = base + path
        result = polite_get(url, insecure=args.insecure)
        evidence_name = "exposed_path_" + path.strip("/").replace("/", "_").replace(".", "_")
        save_evidence(evidence_name, result)

        status = result.get("status")
        size = len(result.get("body", ""))
        flag = ""
        if looks_like_hit(result):
            flag = "  <-- review"
            hits.append((path, status, size))
        err = result.get("error")
        suffix = f" error={err}" if err else ""
        print(f"  {path:30s} status={status!s:5} bytes={size:6d}{flag}{suffix}")

    print("\nSummary:")
    if hits:
        for path, status, size in hits:
            print(f"  [review] {path} (status={status}, bytes={size})")
        print(
            "\n  Non-404/expected responses above need manual review before "
            "being written up as findings -- many sites return 200 for "
            "everything (soft-404 shells), which is a false positive here."
        )
    else:
        print("  No non-404 responses among the curated paths.")


# ---------------------------------------------------------------------------
# siteinfo -- robots.txt + sitemap.xml, parsed (external, no dependencies)
# ---------------------------------------------------------------------------

# Deliberately separate from `paths` above: robots.txt and sitemap.xml are
# meant to be public and machine-read (every search-engine crawler fetches
# them, unauthenticated, all the time) -- unlike the rest of CANDIDATE_PATHS
# (.env, .git/HEAD, admin panels, etc.), a 200 here is the normal/intended
# response, not a "hit" to flag for review. So instead of just reporting
# status/size like `paths` does, this parses the actual content -- if and
# only if the response is 200. Still GET-only, still evidence-saved, still
# gated by require_authorized_domain like everything else external here.

SITEINFO_MAX_ENTRIES_SHOWN = 50


def parse_robots_txt(body):
    """Line-based parse of a robots.txt body -- not a full RFC 9309 parser
    (no group-matching against a specific user-agent), just extraction of
    every directive line for a human to review. Returns a dict of
    directive name (lowercased) -> list of values in file order, plus the
    raw (name, value) pairs for reference."""
    directives = {}
    lines = []
    for raw_line in body.splitlines():
        line = raw_line.split("#", 1)[0].strip()  # strip inline/whole-line comments
        if not line or ":" not in line:
            continue
        name, _, value = line.partition(":")
        name = name.strip().lower()
        value = value.strip()
        if not value:
            continue
        directives.setdefault(name, []).append(value)
        lines.append((name, value))
    return {"directives": directives, "lines": lines}


def parse_sitemap_xml(body):
    """Parse a sitemap.xml body: either a <urlset> of <url><loc> entries, or
    a <sitemapindex> of <sitemap><loc> entries pointing at sub-sitemaps.
    Namespace-agnostic (matches on the local tag name, ignoring any
    '{namespace}' prefix ElementTree attaches) since some servers omit or
    vary the declared sitemaps.org namespace. Returns {"error": ...} instead
    of raising if the body isn't well-formed XML -- a malformed or empty
    response here is normal (plenty of sites don't have a sitemap at all)
    and shouldn't crash the run."""
    try:
        root = ET.fromstring(body)
    except ET.ParseError as e:
        return {"error": f"ParseError: {e}"}

    def local_tag(elem):
        return elem.tag.rsplit("}", 1)[-1]

    root_tag = local_tag(root)
    locs = [
        child.text.strip()
        for child in root.iter()
        if local_tag(child) == "loc" and (child.text or "").strip()
    ]

    return {
        "root_tag": root_tag,
        "is_index": root_tag == "sitemapindex",
        "entry_count": len(locs),
        "entries": locs,
    }


def cmd_siteinfo(args):
    print_banner()
    base = require_authorized_domain(args.base_url.rstrip("/"), label="target")

    print(f"[*] Fetching robots.txt and sitemap.xml from {base}\n", file=sys.stderr)
    print(
        "Unlike `paths`, a 200 here is the normal/intended response for both "
        "files (they're meant to be public and machine-read) -- this parses "
        "the content instead of just flagging presence.\n",
        file=sys.stderr,
    )

    results = {}
    for name, path, parser in (
        ("robots.txt", "/robots.txt", parse_robots_txt),
        ("sitemap.xml", "/sitemap.xml", parse_sitemap_xml),
    ):
        url = base + path
        result = polite_get(url, insecure=args.insecure)
        save_evidence("siteinfo_" + name.replace(".", "_"), result)
        results[name] = {"fetch": result}

        status = result.get("status")
        print(f"{name}: status={status}")
        if result.get("error"):
            print(f"  error={result['error']}")
            print()
            continue
        if status != 200:
            print("  not 200 -- skipping parse (unremarkable; plenty of sites omit either file)")
            print()
            continue

        parsed = parser(result.get("body", ""))
        results[name]["parsed"] = parsed
        if "error" in parsed:
            print(f"  200 but failed to parse: {parsed['error']}")
            print()
            continue

        if name == "robots.txt":
            directives = parsed["directives"]
            disallow = directives.get("disallow", [])
            sitemap_hints = directives.get("sitemap", [])
            plural = "y" if len(disallow) == 1 else "ies"
            print(f"  {len(parsed['lines'])} directive line(s), {len(disallow)} Disallow entr{plural}")
            for entry in disallow[:SITEINFO_MAX_ENTRIES_SHOWN]:
                print(f"    Disallow: {entry}")
            if len(disallow) > SITEINFO_MAX_ENTRIES_SHOWN:
                print(f"    ... and {len(disallow) - SITEINFO_MAX_ENTRIES_SHOWN} more (see evidence/)")
            if sitemap_hints:
                print(f"  Sitemap: directive(s) found inside robots.txt: {sitemap_hints}")
        else:  # sitemap.xml
            kind = "sitemap index (references other sitemaps)" if parsed["is_index"] else "urlset"
            plural = "y" if parsed["entry_count"] == 1 else "ies"
            print(f"  {kind}, {parsed['entry_count']} <loc> entr{plural}")
            for loc in parsed["entries"][:SITEINFO_MAX_ENTRIES_SHOWN]:
                print(f"    {loc}")
            if parsed["entry_count"] > SITEINFO_MAX_ENTRIES_SHOWN:
                print(f"    ... and {parsed['entry_count'] - SITEINFO_MAX_ENTRIES_SHOWN} more (see evidence/)")
        print()

    save_evidence("siteinfo_summary", results)
    print(
        "Neither file is itself a finding -- this command doesn't flag "
        "anything. Disallow entries or sitemap URLs that point at something "
        "unexpected (admin/staging/internal-sounding paths) are worth a "
        "manual look and a findings/ write-up if so.",
        file=sys.stderr,
    )


# ---------------------------------------------------------------------------
# adpaths -- DNS -> SUBDOMAINS -> PATHS chain, looking for the AD/internal
# pivot point (external, sends real GETs to discovered subdomains)
# ---------------------------------------------------------------------------

# scope.md's "Internal network access path" is explicitly "via the external
# website -- i.e. the internal AD/SMB foothold is expected to be reached by
# pivoting from the external web application", and puts active testing of
# the external web app in scope "specifically to the extent needed to
# establish that internal foothold". This chains dns's subdomain discovery
# into a curated, read-only GET check across every *discovered, live*
# subdomain of an authorized domain, for paths that typically indicate an
# Active-Directory-integrated or remote-access endpoint -- candidate pivot
# points, not a general disclosure-path fuzzer. GET-only, curated list, no
# auth attempts, no CVE-specific requests.
AD_PIVOT_PATHS = [
    ("/owa/", "Outlook Web App -- Exchange, almost always AD-integrated"),
    ("/ecp/", "Exchange Control Panel -- Exchange admin, AD-integrated"),
    ("/autodiscover/autodiscover.xml", "Exchange/Outlook autodiscover -- confirms AD-integrated mail"),
    ("/adfs/ls/", "AD FS login service -- direct AD Federation Services signal"),
    ("/adfs/services/trust/2005/usernamemixed", "AD FS WS-Trust endpoint -- direct ADFS signal"),
    ("/remote/login", "SSL-VPN portal login (e.g. FortiGate) -- known remote-access entry point"),
    ("/vpn/index.html", "VPN portal marker (e.g. Citrix Gateway)"),
    ("/rdweb/", "RD Web Access -- Remote Desktop Services, AD-integrated"),
    ("/citrix/", "Citrix StoreFront/Gateway -- often AD-integrated auth"),
    ("/certsrv/", "AD Certificate Services web enrollment -- strong AD signal"),
    # SharePoint-specific. No recon has confirmed Teva runs SharePoint
    # anywhere -- these are included speculatively as part of the same
    # generic curated list as the Exchange/ADFS/VPN-appliance entries above,
    # not because anything has been observed yet. On-prem SharePoint
    # (unlike SharePoint Online) is typically AD-integrated via Windows
    # Authentication (NTLM/Kerberos) or ADFS claims auth, which is why
    # these paths are useful to have in the curated list at all.
    ("/_vti_inf.html", "SharePoint/FrontPage Server Extensions version-disclosure file -- if present, body reveals the exact server version"),
    ("/_vti_bin/sitedata.asmx", "SharePoint SOAP web service -- presence confirms SharePoint"),
    ("/_layouts/15/authenticate.aspx", "SharePoint sign-in page -- an unauthenticated 401 here often carries a WWW-Authenticate: NTLM/Negotiate header"),
    ("/_api/web", "SharePoint REST API root -- presence/version signal, may also trigger a Windows-Auth challenge"),
    ("/+CSCOE+/logon.html", "Cisco ASA/AnyConnect WebVPN login"),
    ("/global-protect/login.esp", "Palo Alto GlobalProtect VPN login"),
]


def discover_live_hosts(domain):
    """DNS -> SUBDOMAINS: resolve the base domain, the curated candidate
    list, and any additional names the configured certificate-transparency
    / passive-recon sources (see discover_ct_names) have logged for it, and
    return only the ones that actually resolve AND aren't explicitly
    excluded (see EXCLUDED_HOSTS -- for hosts confirmed out of scope per
    scope.md even though they're a subdomain of an authorized domain, e.g.
    third-party-hosted infrastructure). Every hostname returned is, by
    construction, `domain` itself or a subdomain of it: candidates are
    built as "<label>.<domain>", and CT-source names are filtered through
    is_authorized_domain() (which also excludes EXCLUDED_HOSTS) before
    being resolved -- the caller must already have validated `domain`
    itself via require_authorized_domain()."""
    live = []
    if not is_excluded_host(domain) and dns_resolve(domain)["resolved"]:
        live.append(domain)

    for sub in CANDIDATE_SUBDOMAINS:
        hostname = f"{sub}.{domain}"
        if is_excluded_host(hostname):
            continue
        if dns_resolve(hostname)["resolved"]:
            live.append(hostname)

    ct = discover_ct_names(domain)
    for name in ct["names"]:
        name = name.lstrip("*.")
        if name in live or not is_authorized_domain(name):
            continue
        if dns_resolve(name)["resolved"]:
            live.append(name)

    return sorted(set(live))


def cmd_adpaths(args):
    print_banner()
    domain = args.domain
    if domain is None:
        if not AUTHORIZED_DOMAINS:
            print(
                "usage: adpaths <domain> -- AUTHORIZED_DOMAINS is empty, so there is "
                "nothing to default to. Resolve ../scope.md's open items and add a "
                "domain there first.",
                file=sys.stderr,
            )
            sys.exit(2)
        if len(AUTHORIZED_DOMAINS) != 1:
            print(
                "usage: adpaths <domain> -- multiple AUTHORIZED_DOMAINS are "
                f"configured, pick one explicitly: {', '.join(AUTHORIZED_DOMAINS)}",
                file=sys.stderr,
            )
            sys.exit(2)
        domain = AUTHORIZED_DOMAINS[0]
    require_authorized_domain(domain, label="domain")

    print(
        f"[*] DNS -> SUBDOMAINS -> PATHS: discovering live hosts under {domain}, "
        f"then checking {len(AD_PIVOT_PATHS)} curated AD/SSO/remote-access "
        "paths on each -- read-only GETs looking for the internal-network "
        "pivot point scope.md describes, not a disclosure fuzzer.\n",
        file=sys.stderr,
    )
    if args.insecure:
        print(
            "[*] --insecure: TLS certificate verification is DISABLED for these "
            "requests. This only affects whether your client trusts the cert; "
            "it does not change anything about the target.",
            file=sys.stderr,
        )

    hosts = discover_live_hosts(domain)
    print(f"Live hosts discovered: {len(hosts)}")
    for h in hosts:
        print(f"  {h}")
    print()

    all_hits = []
    for host in hosts:
        base_url = f"https://{host}"
        for path, reason in AD_PIVOT_PATHS:
            url = base_url + path
            result = polite_get(url, insecure=args.insecure)
            evidence_name = "adpaths_" + host.replace(".", "_") + "_" + path.strip("/").replace("/", "_").replace(".", "_")
            save_evidence(evidence_name, result)

            status = result.get("status")
            size = len(result.get("body", ""))
            hit = looks_like_hit(result)
            flag = "  <-- possible AD pivot point" if hit else ""
            err = result.get("error")
            suffix = f" error={err}" if err else ""
            label = host + path
            print(f"  {label:60s} status={status!s:5} bytes={size:6d}{flag}{suffix}")
            if hit:
                all_hits.append((host, path, status, size, reason))

    print("\nSummary:")
    if all_hits:
        for host, path, status, size, reason in all_hits:
            print(f"  [review] https://{host}{path} (status={status}, bytes={size}) -- {reason}")
    else:
        print("  No non-404 responses among the curated AD-pivot paths on any live host.")

    print(
        "\nGET-only, curated list, no auth attempts, no CVE-specific requests. "
        "A hit here is a CANDIDATE internal-network pivot point per scope.md's "
        "\"internal access via the external website\" section -- confirm by "
        "hand and update scope.md/recon/ before treating it as the actual "
        "access path.",
        file=sys.stderr,
    )


# ---------------------------------------------------------------------------
# ntlminfo -- passive NTLM auth-challenge domain-name disclosure (one URL,
# two GETs: bare + a standard NTLM negotiate header)
# ---------------------------------------------------------------------------

# If a discovered endpoint offers Windows Integrated Authentication
# (NTLM) -- e.g. an on-prem SharePoint, Exchange/OWA, or similar
# AD-integrated app found via `adpaths` -- its Type 2 CHALLENGE_MESSAGE
# response includes the internal NetBIOS/DNS domain name as protocol
# metadata -- sent to ANY client during the standard, pre-authentication
# handshake negotiation, not a secret. Reading it requires sending a Type 1
# NEGOTIATE_MESSAGE (a fixed, standard first message -- the same bytes any
# NTLM-aware HTTP client sends, carrying no credential of any kind) and
# reading the Type 2 reply. This stops there: no Type 3 message is ever
# built or sent, so no authentication is attempted and no password/hash is
# used or guessed -- this is how scope.md's "internal AD domain name
# unknown" open item (if internal testing is even in scope) would get
# resolved, without an auth attempt.
#
# Known limitation: polite_get() stores response headers as a dict built
# from resp.getheaders(), which collapses multiple same-named header lines
# to the last one seen. A server offering both "WWW-Authenticate: Negotiate"
# and "WWW-Authenticate: NTLM" as separate header lines will still work here
# in the common IIS ordering (Negotiate listed first, NTLM second -- so NTLM
# survives the collapse), but this isn't guaranteed. If a target clearly
# supports NTLM but this command reports otherwise, that's the likely cause.

NTLM_AV_PAIR_NAMES = {
    1: "netbios_computer_name",
    2: "netbios_domain_name",
    3: "dns_computer_name",
    4: "dns_domain_name",
    5: "dns_tree_name",
}


def build_ntlm_negotiate_message():
    """Build a minimal NTLM Type 1 NEGOTIATE_MESSAGE (MS-NLMP 2.2.1.1) with
    no domain/workstation name supplied -- the same fixed, standard first
    message any NTLM-aware HTTP client sends before any real authentication.
    Carries no credential of any kind, just a capability negotiation."""
    signature = b"NTLMSSP\x00"
    message_type = struct.pack("<I", 1)
    # Unicode | OEM | Request Target | NTLM | Always Sign | Extended Session
    # Security -- a conventional minimal set for triggering a Type 2 reply;
    # these bits don't need to match a real Windows client exactly, only
    # what the server needs to see to respond usefully.
    negotiate_flags = struct.pack("<I", 0x00088207)
    domain_fields = struct.pack("<HHI", 0, 0, 32)
    workstation_fields = struct.pack("<HHI", 0, 0, 32)
    return signature + message_type + negotiate_flags + domain_fields + workstation_fields


def parse_ntlm_target_info(data):
    """Parse an NTLM AV_PAIR sequence (MS-NLMP 2.2.2.1) -- returns a dict of
    the text-valued fields (NetBIOS/DNS domain and computer names). Ignores
    AV IDs it doesn't recognize (timestamp, flags, channel bindings, etc)."""
    result = {}
    pos = 0
    while pos + 4 <= len(data):
        av_id, av_len = struct.unpack_from("<HH", data, pos)
        pos += 4
        if av_id == 0:  # MsvAvEOL
            break
        value = data[pos:pos + av_len]
        pos += av_len
        name = NTLM_AV_PAIR_NAMES.get(av_id)
        if name:
            try:
                result[name] = value.decode("utf-16-le")
            except UnicodeDecodeError:
                result[name] = repr(value)
    return result


def parse_ntlm_challenge_message(data):
    """Parse an NTLM Type 2 CHALLENGE_MESSAGE (MS-NLMP 2.2.1.2). Returns a
    dict with target_name (the NetBIOS domain/server name being challenged)
    and target_info (dict from parse_ntlm_target_info, empty if the message
    didn't include one -- older non-NTLMv2 servers omit it). Raises
    ValueError for anything that doesn't look like a Type 2 message."""
    if len(data) < 32 or data[:8] != b"NTLMSSP\x00":
        raise ValueError("not an NTLMSSP message (bad signature)")
    message_type = struct.unpack_from("<I", data, 8)[0]
    if message_type != 2:
        raise ValueError(f"not a Type 2 message (got type {message_type})")

    target_name_len, _, target_name_offset = struct.unpack_from("<HHI", data, 12)
    negotiate_flags = struct.unpack_from("<I", data, 20)[0]
    unicode_names = bool(negotiate_flags & 0x1)

    target_name_bytes = data[target_name_offset:target_name_offset + target_name_len]
    try:
        target_name = target_name_bytes.decode("utf-16-le" if unicode_names else "latin-1")
    except UnicodeDecodeError:
        target_name = repr(target_name_bytes)

    target_info = {}
    if len(data) >= 48:
        target_info_len, _, target_info_offset = struct.unpack_from("<HHI", data, 40)
        if target_info_len and target_info_offset:
            target_info = parse_ntlm_target_info(data[target_info_offset:target_info_offset + target_info_len])

    return {"target_name": target_name, "negotiate_flags": negotiate_flags, "target_info": target_info}


def get_header_ci(headers, name):
    """Case-insensitive header lookup -- HTTP header names are
    case-insensitive but polite_get's dict preserves whatever case the
    server sent."""
    name_lower = name.lower()
    for k, v in headers.items():
        if k.lower() == name_lower:
            return v
    return None


def cmd_ntlminfo(args):
    print_banner()
    url = require_authorized_domain(args.url, label="target")

    print(
        f"[*] NTLM auth-challenge probe: {url}\n"
        "Two GETs only -- one bare, one with a standard NTLM Type 1 negotiate "
        "header (the same first message any NTLM-aware HTTP client sends "
        "before any real authentication). No credentials of any kind are "
        "sent or guessed; this stops as soon as the server's Type 2 "
        "challenge is read, well before a real auth attempt (Type 3) would "
        "happen.\n",
        file=sys.stderr,
    )

    baseline = polite_get(url, insecure=args.insecure)
    save_evidence("ntlminfo_baseline", baseline)
    baseline_auth = get_header_ci(baseline.get("headers", {}), "WWW-Authenticate")
    print(f"baseline: status={baseline.get('status')} WWW-Authenticate={baseline_auth!r}")
    if baseline.get("error"):
        print(f"  error={baseline['error']}")

    if not baseline_auth or "ntlm" not in baseline_auth.lower():
        print(
            "\nNo NTLM offered in the baseline response's WWW-Authenticate "
            "header -- nothing to negotiate. (Negotiate/Kerberos-only "
            "endpoints, where the server never falls back to bare NTLM, "
            "aren't supported by this probe.)",
            file=sys.stderr,
        )
        return

    negotiate_b64 = base64.b64encode(build_ntlm_negotiate_message()).decode("ascii")
    challenge_resp = polite_get(url, insecure=args.insecure, extra_headers={"Authorization": f"NTLM {negotiate_b64}"})
    save_evidence("ntlminfo_challenge", challenge_resp)

    challenge_auth = get_header_ci(challenge_resp.get("headers", {}), "WWW-Authenticate")
    print(f"\nafter NTLM negotiate: status={challenge_resp.get('status')}")
    if challenge_resp.get("error"):
        print(f"  error={challenge_resp['error']}")

    if not challenge_auth or "ntlm" not in challenge_auth.lower():
        print(
            "No NTLM challenge came back -- the server may require session "
            "state from the first request, wrap NTLM inside Negotiate/SPNEGO "
            "(not unwrapped here), or not actually support NTLM despite the "
            "baseline header.",
            file=sys.stderr,
        )
        return

    parts = challenge_auth.split(None, 1)
    b64_challenge = parts[1].strip() if len(parts) == 2 else None
    if not b64_challenge:
        print("Could not find a base64 payload after \"NTLM\" in the WWW-Authenticate header.", file=sys.stderr)
        return

    try:
        parsed = parse_ntlm_challenge_message(base64.b64decode(b64_challenge))
    except Exception as e:  # noqa: BLE001 -- report a malformed/unexpected message as evidence, not a crash
        print(f"Failed to parse the NTLM challenge message: {type(e).__name__}: {e}", file=sys.stderr)
        return

    print("\nDecoded NTLM Type 2 challenge:")
    print(f"  target_name (NetBIOS domain/server): {parsed['target_name']!r}")
    for key in ("netbios_domain_name", "netbios_computer_name", "dns_domain_name", "dns_computer_name", "dns_tree_name"):
        if key in parsed["target_info"]:
            print(f"  {key}: {parsed['target_info'][key]!r}")
    if not parsed["target_info"]:
        print("  (no TargetInfo block in this message -- only target_name above is available)")

    print(
        "\nThis is metadata the server volunteers to any client during "
        "standard auth negotiation, not a credential attack. No "
        "authentication was completed -- no Type 3 message was sent, no "
        "password or hash was used or guessed.",
        file=sys.stderr,
    )


# ---------------------------------------------------------------------------
# cve -- static, offline known-CVE reference lookup (NO network access)
# ---------------------------------------------------------------------------

# Hardcoded reference data only -- publicly documented vendor PSIRT advisories
# / NVD entries for the remote-access product families `vpn` above can flag.
# This section makes NO network request of any kind (not to the target, not
# to NVD, not to any third party) and sends NOTHING to any host -- it exists
# so that scope.md's open item "(b) checked against known CVEs for that exact
# version" can be satisfied WITHOUT the live CVE-specific probe scope.md
# explicitly forbids until sign-off (e.g. the CVE-2018-13379 request pattern).
#
# ACCURACY NOTE: affected-version ranges reflect public reporting at the time
# this table was written. The exact trailing patch-level cutoff (the last
# affected build in a branch) is the part most likely to be imprecise from
# memory -- always confirm the authoritative cutoff against the advisory ID
# / NVD link before relying on a "not affected" read. A "not in range" result
# here is a starting hint, never a confirmed-safe verdict.
CVE_DATABASE = {
    "fortinet": [
        {
            "id": "CVE-2018-13379",
            "summary": (
                "FortiOS SSL-VPN web portal path traversal allowing pre-auth "
                "disclosure of session files, which can include cached "
                "plaintext credentials. One of the most widely exploited VPN "
                "CVEs in ransomware intrusions circa 2020-2021."
            ),
            "pre_auth": True,
            "severity": "Critical",
            "affected_versions_text": "FortiOS 6.0.0-6.0.4, 5.6.3-5.6.7, 5.4.6-5.4.12",
            "version_ranges": [((6, 0, 0), (6, 0, 4)), ((5, 6, 3), (5, 6, 7)), ((5, 4, 6), (5, 4, 12))],
            "fixed_versions": "6.0.5, 5.6.8, 5.4.13 or later",
            "advisory": "Fortinet PSIRT FG-IR-18-384",
            "reference": "https://nvd.nist.gov/vuln/detail/CVE-2018-13379",
        },
        {
            "id": "CVE-2022-42475",
            "summary": (
                "FortiOS/FortiProxy SSL-VPN heap-based buffer overflow "
                "allowing pre-auth remote code execution."
            ),
            "pre_auth": True,
            "severity": "Critical",
            "affected_versions_text": (
                "FortiOS 7.2.0-7.2.2, 7.0.0-7.0.8, 6.4.0-6.4.10, 6.2.0-6.2.11, "
                "6.0.0-6.0.15 (approximate upper bounds -- confirm exact "
                "cutoff against the advisory)"
            ),
            "version_ranges": [
                ((7, 2, 0), (7, 2, 2)), ((7, 0, 0), (7, 0, 8)), ((6, 4, 0), (6, 4, 10)),
                ((6, 2, 0), (6, 2, 11)), ((6, 0, 0), (6, 0, 15)),
            ],
            "fixed_versions": "7.2.3, 7.0.9, 6.4.11, 6.2.12, 6.0.16 or later",
            "advisory": "Fortinet PSIRT FG-IR-22-398",
            "reference": "https://nvd.nist.gov/vuln/detail/CVE-2022-42475",
        },
        {
            "id": "CVE-2023-27997",
            "summary": (
                "FortiOS/FortiProxy SSL-VPN heap-based buffer overflow "
                "(\"XORtigate\") allowing pre-auth remote code execution."
            ),
            "pre_auth": True,
            "severity": "Critical",
            "affected_versions_text": (
                "FortiOS 7.2.0-7.2.4, 7.0.0-7.0.11, 6.4.0-6.4.12, 6.2.0-6.2.12, "
                "6.0.0-6.0.17 (approximate upper bounds -- confirm exact "
                "cutoff against the advisory)"
            ),
            "version_ranges": [
                ((7, 2, 0), (7, 2, 4)), ((7, 0, 0), (7, 0, 11)), ((6, 4, 0), (6, 4, 12)),
                ((6, 2, 0), (6, 2, 12)), ((6, 0, 0), (6, 0, 17)),
            ],
            "fixed_versions": "7.2.5, 7.0.12, 6.4.13, 6.2.13, 6.0.18 or later",
            "advisory": "Fortinet PSIRT FG-IR-23-097",
            "reference": "https://nvd.nist.gov/vuln/detail/CVE-2023-27997",
        },
    ],
    "ivanti": [
        {
            "id": "CVE-2019-11510",
            "summary": "Pulse Connect Secure arbitrary pre-auth file read, "
                       "widely used to steal credentials/session state.",
            "pre_auth": True,
            "severity": "Critical",
            "affected_versions_text": "Pulse Connect Secure 8.2 before 8.2R12.1, 8.3 before 8.3R7.1, 9.0 before 9.0R3.4",
            "version_ranges": [],
            "fixed_versions": "8.2R12.1, 8.3R7.1, 9.0R3.4 or later",
            "advisory": "Ivanti/Pulse Secure SA44101",
            "reference": "https://nvd.nist.gov/vuln/detail/CVE-2019-11510",
        },
        {
            "id": "CVE-2023-46805",
            "summary": "Ivanti Connect Secure / Policy Secure authentication "
                       "bypass in a web component; chainable with "
                       "CVE-2024-21887 for unauthenticated RCE.",
            "pre_auth": True,
            "severity": "Critical",
            "affected_versions_text": (
                "Ivanti Connect Secure 9.x and 22.x, Ivanti Policy Secure -- "
                "any build not confirmed to include Ivanti's January 2024 "
                "security patches; per-branch patched build numbers are "
                "numerous -- check Ivanti's bulletin directly rather than a "
                "specific number here."
            ),
            "version_ranges": [],
            "fixed_versions": "see Ivanti's January 2024 security advisory for per-branch patched builds",
            "advisory": "Ivanti Security Advisory (Jan 2024)",
            "reference": "https://nvd.nist.gov/vuln/detail/CVE-2023-46805",
        },
        {
            "id": "CVE-2024-21887",
            "summary": "Command injection in Ivanti Connect Secure / Policy "
                       "Secure web components; chained with CVE-2023-46805 "
                       "for unauthenticated RCE (exploited in the wild as a "
                       "0-day chain).",
            "pre_auth": True,
            "severity": "Critical",
            "affected_versions_text": "Ivanti Connect Secure 9.x/22.x, Ivanti Policy Secure -- same caveat as CVE-2023-46805 above.",
            "version_ranges": [],
            "fixed_versions": "see Ivanti's January 2024 security advisory",
            "advisory": "Ivanti Security Advisory (Jan 2024)",
            "reference": "https://nvd.nist.gov/vuln/detail/CVE-2024-21887",
        },
        {
            "id": "CVE-2024-21888",
            "summary": "Privilege escalation in a web component of Ivanti "
                       "Connect Secure and Ivanti Policy Secure.",
            "pre_auth": False,
            "severity": "High",
            "affected_versions_text": "Ivanti Connect Secure 9.x/22.x, Ivanti Policy Secure -- same caveat as CVE-2023-46805 above.",
            "version_ranges": [],
            "fixed_versions": "see Ivanti's January 2024 security advisory",
            "advisory": "Ivanti Security Advisory (Jan 2024)",
            "reference": "https://nvd.nist.gov/vuln/detail/CVE-2024-21888",
        },
        {
            "id": "CVE-2024-21893",
            "summary": "SSRF in the SAML component of Ivanti Connect Secure, "
                       "Policy Secure, and Neurons for ZTA gateways -- "
                       "exploited in the wild as a follow-on 0-day shortly "
                       "after the Jan 2024 patch for the CVE-2023-46805/"
                       "CVE-2024-21887 pair.",
            "pre_auth": True,
            "severity": "Critical",
            "affected_versions_text": "Ivanti Connect Secure 9.x/22.x, Policy Secure, Neurons for ZTA gateways -- check Ivanti's bulletin for per-branch patched builds.",
            "version_ranges": [],
            "fixed_versions": "see Ivanti's follow-up security advisory",
            "advisory": "Ivanti Security Advisory (follow-up, Jan 2024)",
            "reference": "https://nvd.nist.gov/vuln/detail/CVE-2024-21893",
        },
    ],
    "citrix": [
        {
            "id": "CVE-2019-19781",
            "summary": "Citrix ADC/Gateway (\"Shitrix\") path traversal "
                       "leading to pre-auth remote code execution.",
            "pre_auth": True,
            "severity": "Critical",
            "affected_versions_text": (
                "Citrix ADC/Gateway before 11.1.51.21, 12.0.53.13, 12.1.49.23, "
                "13.0.47.24; Citrix SD-WAN WANOP 4000-WO/4100-WO/5000-WO/"
                "5100-WO before 10.2.6b/11.0.3b (approximate -- confirm exact "
                "builds against the advisory)."
            ),
            "version_ranges": [],
            "fixed_versions": "see Citrix advisory CTX267027",
            "advisory": "Citrix CTX267027",
            "reference": "https://nvd.nist.gov/vuln/detail/CVE-2019-19781",
        },
        {
            "id": "CVE-2023-3519",
            "summary": "Citrix NetScaler ADC/Gateway unauthenticated remote "
                       "code execution.",
            "pre_auth": True,
            "severity": "Critical",
            "affected_versions_text": (
                "NetScaler ADC/Gateway 13.1 before 13.1-49.13, 13.0 before "
                "13.0-91.13, 13.1-FIPS before 13.1-37.159, 12.1-FIPS/"
                "12.1-NDcPP before 12.1-55.297; 12.1 is EOL -- treat as "
                "affected regardless of build (approximate -- confirm "
                "against the advisory)."
            ),
            "version_ranges": [],
            "fixed_versions": "see Citrix advisory CTX561482",
            "advisory": "Citrix CTX561482",
            "reference": "https://nvd.nist.gov/vuln/detail/CVE-2023-3519",
        },
    ],
}


def parse_dotted_version(s):
    """Parse a plain dotted numeric version string (e.g. "6.4.6") into a tuple
    of ints for comparison. Returns None if it doesn't look like one -- some
    product families (e.g. Ivanti's "9.1R15.2") don't use this scheme at all
    and are deliberately not auto-parsed here."""
    try:
        return tuple(int(p) for p in s.strip().split("."))
    except ValueError:
        return None


def fortios_version_ready_for_range_check(version_tuple):
    """True only for a complete MAJOR.MINOR.PATCH tuple (exactly 3 parts).

    Every range in CVE_DATABASE is a 3-tuple, and Python happily compares
    tuples of different lengths (e.g. (7, 2) < (7, 2, 0) is True) -- without
    this check, a partial version like "7.2" would silently fail every
    range check and report zero matches, a false "not affected" for e.g.
    7.2.0, which IS in-range for two CVEs below."""
    return version_tuple is not None and len(version_tuple) == 3


def fortios_entry_matches(version_tuple, entry):
    """True if version_tuple falls within any of entry's version_ranges.
    Caller must have already confirmed
    fortios_version_ready_for_range_check(version_tuple)."""
    return any(low <= version_tuple <= high for low, high in entry["version_ranges"])


def fortios_cve_matches(version_tuple, entries):
    """Ids of entries whose version_ranges contain version_tuple. Caller must
    have already confirmed fortios_version_ready_for_range_check(version_tuple)."""
    return [entry["id"] for entry in entries if fortios_entry_matches(version_tuple, entry)]


def cmd_cve(args):
    print_banner()
    print(
        "[*] Static, offline CVE reference lookup -- this makes NO network "
        "request of any kind (not to the target, not to NVD, not anywhere). "
        "It is a hardcoded table of publicly documented advisories, "
        "optionally cross-checked against a version string YOU supply. It "
        "does not detect, confirm, or probe anything on a live target. If "
        "you don't yet have a confirmed version, get one legitimately (ask "
        "the vendor/target contact directly -- see scope.md) rather than "
        "probing for it.\n",
        file=sys.stderr,
    )

    entries = CVE_DATABASE.get(args.product, [])
    if not entries:
        print(f"No entries on file for product {args.product!r}.", file=sys.stderr)
        return

    version_tuple = parse_dotted_version(args.version) if args.version else None
    can_range_check = args.product == "fortinet" and fortios_version_ready_for_range_check(version_tuple)
    if args.version and args.product == "fortinet" and not can_range_check:
        print(
            f"warning: {args.version!r} doesn't look like a complete "
            "MAJOR.MINOR.PATCH FortiOS version (e.g. 6.4.6) -- listing all "
            "known CVEs without range matching, rather than risk a false "
            "'not affected' from a partial version.\n",
            file=sys.stderr,
        )

    print(f"Known publicly-documented CVEs for product family: {args.product}\n")
    report_cves = []
    for entry in entries:
        line_status = None
        if can_range_check:
            in_range = fortios_entry_matches(version_tuple, entry)
            line_status = (
                "POSSIBLY AFFECTED -- version falls in a range on file (hint "
                "only, verify against the advisory before acting)"
                if in_range
                else "outside the ranges on file (NOT a confirmed-safe result "
                "-- range data may be stale/imprecise; verify against the "
                "live advisory)"
            )

        pre_auth_tag = ", pre-auth" if entry["pre_auth"] else ""
        print(f"  {entry['id']}  [{entry['severity']}{pre_auth_tag}]")
        print(f"    {entry['summary']}")
        print(f"    affected: {entry['affected_versions_text']}")
        print(f"    fixed in: {entry['fixed_versions']}")
        print(f"    advisory: {entry['advisory']}   reference: {entry['reference']}")
        if line_status:
            print(f"    --> {line_status}")
        elif args.version:
            print(
                f"    --> automatic version matching not implemented for this "
                f"product's version scheme -- compare {args.version!r} by "
                "hand against 'affected' above and the linked advisory."
            )
        print()
        report_cves.append({**entry, "match_status": line_status})

    print(
        "Reference data only, reconstructed from public advisories at "
        "authoring time -- it can be imprecise at the exact trailing-patch "
        "level and does not substitute for the vendor's live advisory or NVD "
        "entry. Never treat a 'not in range' result as confirmation the "
        "target is safe. This command sent nothing to the target (or "
        "anywhere else) and performed no network access at all.",
        file=sys.stderr,
    )

    save_evidence(
        f"cve_lookup_{args.product}",
        {
            "product": args.product,
            "version_supplied": args.version,
            "network_access": False,
            "note": "static offline reference lookup -- no requests sent to any host",
            "cves": report_cves,
        },
    )


# ---------------------------------------------------------------------------
# selftest -- this file's own embedded offline unit tests (no network, no
# target contact, stdlib only). Embedded here rather than in the repo's
# tests/ directory so the checks travel with the file wherever it's copied
# -- see the "Standalone" note in the module docstring.
# ---------------------------------------------------------------------------

class _ParseDottedVersionTests(unittest.TestCase):
    def test_parses_plain_dotted_version(self):
        self.assertEqual(parse_dotted_version("6.4.6"), (6, 4, 6))

    def test_rejects_non_numeric(self):
        self.assertIsNone(parse_dotted_version("6.x.4"))
        self.assertIsNone(parse_dotted_version("abc"))
        self.assertIsNone(parse_dotted_version("6.4.6-beta"))


class _FortiosRangeCheckEligibilityTests(unittest.TestCase):
    """Regression coverage for a real bug: a partial version like "7.2"
    parses successfully to (7, 2), and Python's tuple comparison silently
    allows comparing a 2-tuple against the 3-tuple ranges (e.g.
    (7, 2) < (7, 2, 0) is True) -- which made every range check fail and
    report zero matches, a false "not affected" for a version (7.2.0) that
    IS in range. Range-checking must require exactly MAJOR.MINOR.PATCH."""

    def test_complete_version_is_eligible(self):
        self.assertTrue(fortios_version_ready_for_range_check((7, 2, 0)))

    def test_partial_major_minor_is_not_eligible(self):
        self.assertFalse(fortios_version_ready_for_range_check((7, 2)))

    def test_major_only_is_not_eligible(self):
        self.assertFalse(fortios_version_ready_for_range_check((7,)))

    def test_four_part_version_is_not_eligible(self):
        self.assertFalse(fortios_version_ready_for_range_check((6, 4, 6, 1)))

    def test_none_is_not_eligible(self):
        self.assertFalse(fortios_version_ready_for_range_check(None))


class _FortiosCveMatchTests(unittest.TestCase):
    def test_known_affected_version_matches_all_three_cves(self):
        # 6.0.0-6.0.4 is the overlap of all three Fortinet ranges on file.
        matches = fortios_cve_matches((6, 0, 2), CVE_DATABASE["fortinet"])
        self.assertEqual(
            set(matches), {"CVE-2018-13379", "CVE-2022-42475", "CVE-2023-27997"}
        )

    def test_version_just_above_a_fix_no_longer_matches_that_cve(self):
        # 5.4.13 is the documented fixed version for CVE-2018-13379.
        matches = fortios_cve_matches((5, 4, 13), CVE_DATABASE["fortinet"])
        self.assertNotIn("CVE-2018-13379", matches)

    def test_version_in_gap_between_branches_matches_nothing(self):
        # 6.1.0 falls between the 6.0.x and 6.2.x branches on file.
        self.assertEqual(fortios_cve_matches((6, 1, 0), CVE_DATABASE["fortinet"]), [])

    def test_partial_version_would_falsely_show_no_match_if_not_guarded(self):
        # Demonstrates *why* the eligibility gate above is required: 7.2.0 is
        # in range for two CVEs, but comparing the partial tuple (7, 2)
        # directly against those same 3-tuple ranges reports zero matches.
        # Callers must check fortios_version_ready_for_range_check() first
        # and refuse to draw a "not affected" conclusion from this shape.
        full = fortios_cve_matches((7, 2, 0), CVE_DATABASE["fortinet"])
        partial = fortios_cve_matches((7, 2), CVE_DATABASE["fortinet"])
        self.assertEqual(set(full), {"CVE-2022-42475", "CVE-2023-27997"})
        self.assertEqual(partial, [])
        self.assertFalse(fortios_version_ready_for_range_check((7, 2)))


class _NtlmNegotiateMessageTests(unittest.TestCase):
    def test_builds_well_formed_type1_message(self):
        msg = build_ntlm_negotiate_message()
        self.assertEqual(msg[:8], b"NTLMSSP\x00")
        self.assertEqual(struct.unpack_from("<I", msg, 8)[0], 1)
        self.assertEqual(len(msg), 32)

    def test_round_trips_through_base64_like_the_real_header_would(self):
        msg = build_ntlm_negotiate_message()
        decoded = base64.b64decode(base64.b64encode(msg).decode("ascii"))
        self.assertEqual(decoded, msg)


class _NtlmChallengeParsingTests(unittest.TestCase):
    """Round-trip tests against a synthetic Type 2 message built by hand
    here (not via build_ntlm_negotiate_message, which builds a Type 1
    message) -- this can't validate against a real Windows server's exact
    byte layout, but it does catch internal bugs in the parser (wrong
    struct format, off-by-one offsets, etc) against the documented MS-NLMP
    CHALLENGE_MESSAGE structure."""

    @staticmethod
    def _build_synthetic_type2_message(target_name="EXAMPLE", av_pairs=None):
        signature = b"NTLMSSP\x00"
        message_type = struct.pack("<I", 2)
        negotiate_flags_value = 0x00820001  # Unicode | Target Info present (illustrative)
        server_challenge = b"\x01\x02\x03\x04\x05\x06\x07\x08"
        reserved = b"\x00" * 8

        target_name_bytes = target_name.encode("utf-16-le")

        av_blob = b""
        if av_pairs:
            for av_id, value in av_pairs:
                value_bytes = value.encode("utf-16-le")
                av_blob += struct.pack("<HH", av_id, len(value_bytes)) + value_bytes
            av_blob += struct.pack("<HH", 0, 0)  # MsvAvEOL terminator

        # Fixed header up through TargetInfoFields (no optional Version block).
        header_len = 8 + 4 + 8 + 4 + 8 + 8 + 8
        target_name_offset = header_len
        target_info_offset = target_name_offset + len(target_name_bytes)

        target_name_fields = struct.pack("<HHI", len(target_name_bytes), len(target_name_bytes), target_name_offset)
        target_info_fields = struct.pack("<HHI", len(av_blob), len(av_blob), target_info_offset)
        negotiate_flags = struct.pack("<I", negotiate_flags_value)

        message = (
            signature + message_type + target_name_fields + negotiate_flags
            + server_challenge + reserved + target_info_fields
        )
        message += target_name_bytes + av_blob
        return message

    def test_parses_target_name_and_domain_info(self):
        data = self._build_synthetic_type2_message(
            target_name="EXAMPLE", av_pairs=[(2, "EXAMPLE"), (4, "example.com"), (1, "DC01")],
        )
        parsed = parse_ntlm_challenge_message(data)
        self.assertEqual(parsed["target_name"], "EXAMPLE")
        self.assertEqual(parsed["target_info"]["netbios_domain_name"], "EXAMPLE")
        self.assertEqual(parsed["target_info"]["dns_domain_name"], "example.com")
        self.assertEqual(parsed["target_info"]["netbios_computer_name"], "DC01")

    def test_rejects_bad_signature(self):
        with self.assertRaises(ValueError):
            parse_ntlm_challenge_message(b"NOTNTLM\x00" + b"\x00" * 40)

    def test_rejects_wrong_message_type(self):
        data = self._build_synthetic_type2_message()
        bad = data[:8] + struct.pack("<I", 1) + data[12:]  # flip type 2 -> 1
        with self.assertRaises(ValueError):
            parse_ntlm_challenge_message(bad)

    def test_handles_missing_target_info(self):
        data = self._build_synthetic_type2_message(target_name="EXAMPLE", av_pairs=None)
        parsed = parse_ntlm_challenge_message(data)
        self.assertEqual(parsed["target_name"], "EXAMPLE")
        self.assertEqual(parsed["target_info"], {})


class _GetHeaderCiTests(unittest.TestCase):
    def test_matches_regardless_of_case(self):
        headers = {"WWW-Authenticate": "NTLM"}
        self.assertEqual(get_header_ci(headers, "www-authenticate"), "NTLM")

    def test_returns_none_when_absent(self):
        self.assertIsNone(get_header_ci({}, "WWW-Authenticate"))


class _CtSourceMergingTests(unittest.TestCase):
    """discover_ct_names must merge across sources and tolerate any subset
    of them failing -- exercised here with monkeypatched sources so this
    doesn't depend on crt.sh/Cert Spotter/Censys actually being reachable
    (selftest is explicitly documented as offline)."""

    @staticmethod
    def _with_sources(crtsh_result, certspotter_result, censys_result):
        class _Restore:
            def __enter__(self):
                self._orig = (
                    globals()["query_crtsh"],
                    globals()["query_certspotter"],
                    globals()["query_censys"],
                )
                globals()["query_crtsh"] = lambda domain: crtsh_result
                globals()["query_certspotter"] = lambda domain: certspotter_result
                globals()["query_censys"] = lambda domain: censys_result

            def __exit__(self, *exc_info):
                globals()["query_crtsh"], globals()["query_certspotter"], globals()["query_censys"] = self._orig

        return _Restore()

    def test_merges_and_deduplicates_across_sources(self):
        with self._with_sources(
            {"ok": True, "names": ["www.example.com", "mail.example.com"]},
            {"ok": True, "names": ["mail.example.com", "vpn.example.com"]},
            {"ok": False, "skipped": True, "error": "no credentials"},
        ):
            result = discover_ct_names("example.com")

        self.assertEqual(result["names"], ["mail.example.com", "vpn.example.com", "www.example.com"])
        self.assertTrue(result["sources"]["crtsh"]["ok"])
        self.assertTrue(result["sources"]["certspotter"]["ok"])
        self.assertTrue(result["sources"]["censys"]["skipped"])

    def test_one_source_failing_does_not_lose_the_others(self):
        with self._with_sources(
            {"ok": False, "error": "Timeout: crt.sh unreachable"},
            {"ok": True, "names": ["vpn.example.com"]},
            {"ok": False, "skipped": True, "error": "no credentials"},
        ):
            result = discover_ct_names("example.com")

        self.assertEqual(result["names"], ["vpn.example.com"])

    def test_censys_skipped_without_credentials(self):
        """Exercises the real query_censys (not monkeypatched) to confirm it
        doesn't attempt a request -- and needs no network access -- when
        CENSYS_API_ID/CENSYS_API_SECRET aren't set."""
        orig_id = os.environ.pop("CENSYS_API_ID", None)
        orig_secret = os.environ.pop("CENSYS_API_SECRET", None)
        try:
            result = query_censys("example.com")
        finally:
            if orig_id is not None:
                os.environ["CENSYS_API_ID"] = orig_id
            if orig_secret is not None:
                os.environ["CENSYS_API_SECRET"] = orig_secret

        self.assertFalse(result["ok"])
        self.assertTrue(result.get("skipped"))


class _RobotsTxtParsingTests(unittest.TestCase):
    def test_extracts_disallow_and_sitemap_directives(self):
        body = (
            "# example robots.txt\n"
            "User-agent: *\n"
            "Disallow: /admin/\n"
            "Disallow: /internal/\n"
            "Allow: /internal/public/\n"
            "\n"
            "Sitemap: https://example.com/sitemap.xml\n"
        )
        parsed = parse_robots_txt(body)
        self.assertEqual(parsed["directives"]["disallow"], ["/admin/", "/internal/"])
        self.assertEqual(parsed["directives"]["allow"], ["/internal/public/"])
        self.assertEqual(parsed["directives"]["sitemap"], ["https://example.com/sitemap.xml"])

    def test_ignores_blank_lines_and_comments(self):
        body = "\n# just a comment\n\n   \nUser-agent: *\n"
        parsed = parse_robots_txt(body)
        self.assertEqual(parsed["directives"], {"user-agent": ["*"]})

    def test_empty_body(self):
        parsed = parse_robots_txt("")
        self.assertEqual(parsed["directives"], {})
        self.assertEqual(parsed["lines"], [])


class _SitemapXmlParsingTests(unittest.TestCase):
    def test_parses_urlset(self):
        body = (
            '<?xml version="1.0" encoding="UTF-8"?>'
            '<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9">'
            "<url><loc>https://example.com/a</loc></url>"
            "<url><loc>https://example.com/b</loc></url>"
            "</urlset>"
        )
        parsed = parse_sitemap_xml(body)
        self.assertFalse(parsed["is_index"])
        self.assertEqual(parsed["entry_count"], 2)
        self.assertEqual(parsed["entries"], ["https://example.com/a", "https://example.com/b"])

    def test_parses_sitemap_index(self):
        body = (
            '<?xml version="1.0" encoding="UTF-8"?>'
            '<sitemapindex xmlns="http://www.sitemaps.org/schemas/sitemap/0.9">'
            "<sitemap><loc>https://example.com/sitemap-1.xml</loc></sitemap>"
            "<sitemap><loc>https://example.com/sitemap-2.xml</loc></sitemap>"
            "</sitemapindex>"
        )
        parsed = parse_sitemap_xml(body)
        self.assertTrue(parsed["is_index"])
        self.assertEqual(parsed["entry_count"], 2)

    def test_parses_without_namespace(self):
        # Some servers serve sitemap.xml without declaring the sitemaps.org
        # namespace at all -- still valid XML, just non-standard.
        body = "<urlset><url><loc>https://example.com/a</loc></url></urlset>"
        parsed = parse_sitemap_xml(body)
        self.assertEqual(parsed["entry_count"], 1)

    def test_malformed_xml_reports_error_not_crash(self):
        parsed = parse_sitemap_xml("<urlset><url><loc>not closed")
        self.assertIn("error", parsed)


class _ScopeExclusionTests(unittest.TestCase):
    """AUTHORIZED_DOMAINS/EXCLUDED_HOSTS change as scope.md's open items get
    resolved (see that file for the current, human-confirmed state). These
    tests exercise the matching logic itself against temporary, test-local
    values (save/restore around each test) instead of depending on
    whatever the production config happens to be right now, so `selftest`
    stays meaningful regardless of how many domains are currently
    configured."""

    @staticmethod
    def _with_domains(authorized, excluded):
        class _Restore:
            def __enter__(self):
                self._orig_authorized = list(AUTHORIZED_DOMAINS)
                self._orig_excluded = list(EXCLUDED_HOSTS)
                AUTHORIZED_DOMAINS[:] = authorized
                EXCLUDED_HOSTS[:] = excluded

            def __exit__(self, *exc_info):
                AUTHORIZED_DOMAINS[:] = self._orig_authorized
                EXCLUDED_HOSTS[:] = self._orig_excluded

        return _Restore()

    def test_no_domains_configured_refuses_everything(self):
        """Regression guard for this scaffold's core safety property: with
        an empty AUTHORIZED_DOMAINS, nothing is authorized -- not even the
        real, currently-configured Teva domains. Exercises the empty case
        directly (via _with_domains) rather than depending on whether
        AUTHORIZED_DOMAINS is currently empty or populated."""
        with self._with_domains([], []):
            self.assertFalse(is_authorized_domain("teva.co.il"))
            self.assertFalse(is_authorized_domain("tevapharm.com"))

    def test_currently_configured_teva_domains_are_authorized(self):
        """Exercises the real module-level AUTHORIZED_DOMAINS (not a
        test-local override) -- both corporate domains the requester named
        (see scope.md) should match themselves, a subdomain, but not an
        unrelated or look-alike host."""
        self.assertTrue(is_authorized_domain("teva.co.il"))
        self.assertTrue(is_authorized_domain("tevapharm.com"))
        self.assertTrue(is_authorized_domain("www.teva.co.il"))
        self.assertFalse(is_authorized_domain("teva.co.il.evil.example"))
        self.assertFalse(is_authorized_domain("not-teva.com"))

    def test_excluded_host_is_not_authorized_even_as_subdomain(self):
        with self._with_domains(["example.com"], ["autodiscover.example.com"]):
            self.assertFalse(is_authorized_domain("autodiscover.example.com"))

    def test_other_subdomains_of_authorized_domain_still_authorized(self):
        with self._with_domains(["example.com"], ["autodiscover.example.com"]):
            self.assertTrue(is_authorized_domain("vpn.example.com"))
            self.assertTrue(is_authorized_domain("example.com"))

    def test_excluded_host_never_appears_in_discover_live_hosts(self):
        # Patches discover_ct_names itself, not the individual query_*
        # source functions it calls -- discover_live_hosts only ever calls
        # discover_ct_names, so patching at that level is what actually
        # keeps this offline (patching only query_crtsh would still let
        # query_certspotter/query_censys attempt real requests).
        def fake_resolve(hostname):
            return {"hostname": hostname, "addresses": ["203.0.113.1"], "resolved": True}

        def fake_discover_ct_names(domain):
            return {"sources": {}, "names": []}

        original_resolve = globals()["dns_resolve"]
        original_discover_ct_names = globals()["discover_ct_names"]
        globals()["dns_resolve"] = fake_resolve
        globals()["discover_ct_names"] = fake_discover_ct_names
        try:
            with self._with_domains(["example.com"], ["autodiscover.example.com"]):
                hosts = discover_live_hosts("example.com")
        finally:
            globals()["dns_resolve"] = original_resolve
            globals()["discover_ct_names"] = original_discover_ct_names

        self.assertNotIn("autodiscover.example.com", hosts)
        self.assertIn("vpn.example.com", hosts)  # sanity: exclusion isn't over-broad


def cmd_selftest(args):
    """Run this file's embedded offline unit tests: pure logic checks over
    CVE_DATABASE and the version helpers above. No network access, no
    target contact, no evidence written."""
    loader = unittest.TestLoader()
    suite = unittest.TestSuite()
    test_classes = (
        _ParseDottedVersionTests,
        _FortiosRangeCheckEligibilityTests,
        _FortiosCveMatchTests,
        _NtlmNegotiateMessageTests,
        _NtlmChallengeParsingTests,
        _GetHeaderCiTests,
        _CtSourceMergingTests,
        _RobotsTxtParsingTests,
        _SitemapXmlParsingTests,
        _ScopeExclusionTests,
    )
    for cls in test_classes:
        suite.addTests(loader.loadTestsFromTestCase(cls))
    result = unittest.TextTestRunner(verbosity=2).run(suite)
    sys.exit(0 if result.wasSuccessful() else 1)


# ---------------------------------------------------------------------------
# smb -- anonymous/null-session SMB enumeration (internal, needs impacket)
# ---------------------------------------------------------------------------

def cmd_smb(args):
    print_banner()
    host = args.host

    try:
        from impacket.smbconnection import SMBConnection
    except ImportError as e:
        print(
            f"Failed to import impacket.smbconnection: {type(e).__name__}: {e}\n"
            "This can mean impacket itself isn't installed, OR it (or a "
            "dependency, e.g. pyasn1/six/pycryptodome/flask/ldap3) is "
            "installed but broken/incompatible with this Python version -- "
            "the message above is the real reason, not just 'not installed'. "
            "Diagnose with:\n"
            "  python3 -c \"from impacket.smbconnection import SMBConnection\"\n"
            "Also confirm you are NOT running this with sudo (see banner).",
            file=sys.stderr,
        )
        sys.exit(1)

    result = {"host": host, "null_session_accepted": False}
    try:
        conn = SMBConnection(host, host, timeout=15)
        try:
            conn.login("", "")  # null session: empty username + password
            result["null_session_accepted"] = True
            result["server_os"] = conn.getServerOS()
            result["server_domain"] = conn.getServerDomain() if hasattr(conn, "getServerDomain") else None

            shares = []
            for share in conn.listShares():
                shares.append(
                    {
                        "name": share["shi1_netname"][:-1] if isinstance(share["shi1_netname"], str) else str(share["shi1_netname"]),
                        "type": share["shi1_type"],
                        "comment": share["shi1_remark"][:-1] if isinstance(share["shi1_remark"], str) else str(share["shi1_remark"]),
                    }
                )
            result["shares"] = shares
        finally:
            conn.close()
    except Exception as e:  # noqa: BLE001 -- report any connection/auth failure as evidence, not a crash
        result["error"] = f"{type(e).__name__}: {e}"

    print(f"host: {host}")
    print(f"null session accepted: {result['null_session_accepted']}")
    if result["null_session_accepted"]:
        print(f"server OS: {result.get('server_os')}")
        print(f"server domain: {result.get('server_domain')}")
        print("shares:")
        for s in result.get("shares", []):
            print(f"  {s['name']:20s} type={s['type']} comment={s['comment']!r}")
    elif "error" in result:
        print(f"error: {result['error']}")

    save_evidence(f"smb_null_session_{host.replace(':', '_').replace('/', '_')}", result)


# ---------------------------------------------------------------------------
# ldap -- anonymous LDAP bind enumeration (internal, needs ldap3)
# ---------------------------------------------------------------------------

def cmd_ldap(args):
    print_banner()
    host = args.host

    try:
        from ldap3 import ALL, Server, Connection
    except ImportError as e:
        print(
            f"Failed to import ldap3: {type(e).__name__}: {e}\n"
            "This can mean ldap3 itself isn't installed, OR it (or a "
            "dependency, e.g. pyasn1) is installed but broken/incompatible "
            "with this Python version. Diagnose with:\n"
            "  python3 -c \"from ldap3 import Server\"\n"
            "Also confirm you are NOT running this with sudo (see banner).",
            file=sys.stderr,
        )
        sys.exit(1)

    result = {"host": host, "anonymous_bind_accepted": False}
    try:
        server = Server(host, get_info=ALL, connect_timeout=15)
        conn = Connection(server)  # anonymous: no user/password supplied
        try:
            bound = conn.bind()
            result["anonymous_bind_accepted"] = bool(bound)
            if bound:
                info = server.info
                result["naming_contexts"] = list(info.naming_contexts) if info else []
                result["supported_ldap_versions"] = list(info.supported_ldap_versions) if info else []
            else:
                result["bind_result"] = str(conn.result)
        finally:
            conn.unbind()
    except Exception as e:  # noqa: BLE001
        result["error"] = f"{type(e).__name__}: {e}"

    print(f"host: {host}")
    print(f"anonymous bind accepted: {result['anonymous_bind_accepted']}")
    if result["anonymous_bind_accepted"]:
        print("naming contexts:")
        for nc in result.get("naming_contexts", []):
            print(f"  {nc}")
    elif "error" in result:
        print(f"error: {result['error']}")
    elif "bind_result" in result:
        print(f"bind result: {result['bind_result']}")

    save_evidence(f"ldap_anonymous_{host.replace(':', '_').replace('/', '_')}", result)


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = parser.add_subparsers(dest="command", required=True)

    p_dns = sub.add_parser(
        "dns", help="passive DNS + multi-source certificate-transparency recon (external, no deps)"
    )
    p_dns.add_argument("domain", nargs="?", default=None,
                        help="which AUTHORIZED_DOMAINS entry to recon -- optional if exactly one is "
                             "configured, required if there's more than one")

    p_siteinfo = sub.add_parser(
        "siteinfo", help="fetch + parse robots.txt and sitemap.xml, only if they 200 (external, no deps)"
    )
    p_siteinfo.add_argument("base_url", help="e.g. https://www.<authorized-domain> -- must match an "
                                              "AUTHORIZED_DOMAINS entry, see scope.md")
    p_siteinfo.add_argument("--insecure", action="store_true",
                             help="skip TLS cert verification (like curl -k) -- client-side only")

    p_paths = sub.add_parser("paths", help="curated disclosure-path check against one host (external, no deps)")
    p_paths.add_argument("base_url", help="e.g. https://www.<authorized-domain> -- must match an "
                                           "AUTHORIZED_DOMAINS entry, see scope.md")
    p_paths.add_argument("--insecure", action="store_true",
                          help="skip TLS cert verification (like curl -k) -- client-side only")

    p_adpaths = sub.add_parser(
        "adpaths",
        help="DNS -> SUBDOMAINS -> PATHS chain: curated AD/SSO-pivot path check across discovered live subdomains (external, no deps)",
    )
    p_adpaths.add_argument("domain", nargs="?", default=None,
                            help="which AUTHORIZED_DOMAINS entry to recon -- optional if exactly one is "
                                 "configured, required if there's more than one")
    p_adpaths.add_argument("--insecure", action="store_true",
                            help="skip TLS cert verification (like curl -k) -- client-side only")

    p_ntlminfo = sub.add_parser(
        "ntlminfo",
        help="passive NTLM auth-challenge domain-name disclosure against one URL (external, no deps)",
    )
    p_ntlminfo.add_argument("url", help="a URL you've observed returning 401 with WWW-Authenticate: NTLM, "
                             "e.g. https://<authorized-domain>/some/observed/401/url")
    p_ntlminfo.add_argument("--insecure", action="store_true",
                             help="skip TLS cert verification (like curl -k) -- client-side only")

    p_cve = sub.add_parser("cve", help="static, offline known-CVE reference lookup -- NO network access (no deps)")
    p_cve.add_argument("--product", required=True, choices=sorted(CVE_DATABASE.keys()),
                        help="product family to look up (e.g. a VPN/remote-access appliance you identified by hand)")
    p_cve.add_argument("--version", help="confirmed version string (e.g. FortiOS '6.4.6') -- obtain this "
                        "legitimately (ask the vendor/target contact); never by probing the target")

    p_smb = sub.add_parser("smb", help="anonymous SMB null-session enum (internal, needs impacket)")
    p_smb.add_argument("host", help="internal host/IP discovered via the authorized access path -- never hardcoded")

    p_ldap = sub.add_parser("ldap", help="anonymous LDAP bind enum (internal, needs ldap3)")
    p_ldap.add_argument("host", help="internal host/IP discovered via the authorized access path -- never hardcoded")

    sub.add_parser("selftest", help="run this file's embedded offline unit tests (no network, no deps)")

    args = parser.parse_args()

    if args.command == "dns":
        cmd_dns(args)
    elif args.command == "siteinfo":
        cmd_siteinfo(args)
    elif args.command == "paths":
        cmd_paths(args)
    elif args.command == "adpaths":
        cmd_adpaths(args)
    elif args.command == "ntlminfo":
        cmd_ntlminfo(args)
    elif args.command == "cve":
        cmd_cve(args)
    elif args.command == "smb":
        cmd_smb(args)
    elif args.command == "ldap":
        cmd_ldap(args)
    elif args.command == "selftest":
        cmd_selftest(args)


if __name__ == "__main__":
    main()


## 3. Safety gate

Flip this only once `pentest-teva/scope.md`'s remaining open items are resolved with real specifics.

In [ ]:
# Set to True only after pentest-teva/scope.md's open items (signed-agreement
# reference, engagement window, authorized source IPs, emergency contact) are
# resolved with real specifics -- not just "yes". Every network-facing cell
# below refuses to run while this is False.
SCOPE_CONFIRMED = False


## 4. Safe to run now (offline, no network)

In [ ]:
!python3 teva_pentest_all.py selftest


In [ ]:
!python3 teva_pentest_all.py cve --product fortinet --version 6.4.6
# swap --product/--version, or drop --version to list everything on file


## 5. Network-facing subcommands -- gated on `SCOPE_CONFIRMED`

Each cell below raises instead of running if `SCOPE_CONFIRMED` is still `False`. Edit the domain/URL/host arguments as needed once you actually run these.

### `dns` -- passive DNS + multi-source certificate-transparency recon

In [ ]:
if not SCOPE_CONFIRMED:
    raise RuntimeError(
        "Refusing to run: SCOPE_CONFIRMED is False. Resolve pentest-teva/scope.md's "
        "open items (signed-agreement reference, engagement window, authorized "
        "source IPs, emergency contact) first, then set SCOPE_CONFIRMED = True above."
    )
!python3 teva_pentest_all.py dns teva.co.il


### `siteinfo` -- fetch + parse robots.txt and sitemap.xml, only if they 200

Unlike `paths` below, a 200 on either file is the normal/expected response, not a hit to flag -- this parses the content.

In [ ]:
if not SCOPE_CONFIRMED:
    raise RuntimeError(
        "Refusing to run: SCOPE_CONFIRMED is False. Resolve pentest-teva/scope.md's "
        "open items (signed-agreement reference, engagement window, authorized "
        "source IPs, emergency contact) first, then set SCOPE_CONFIRMED = True above."
    )
!python3 teva_pentest_all.py siteinfo https://www.teva.co.il


### `paths` -- curated disclosure-path check against one host

In [ ]:
if not SCOPE_CONFIRMED:
    raise RuntimeError(
        "Refusing to run: SCOPE_CONFIRMED is False. Resolve pentest-teva/scope.md's "
        "open items (signed-agreement reference, engagement window, authorized "
        "source IPs, emergency contact) first, then set SCOPE_CONFIRMED = True above."
    )
!python3 teva_pentest_all.py paths https://www.teva.co.il


### `adpaths` -- DNS -> subdomains -> AD/SSO-pivot path check

In [ ]:
if not SCOPE_CONFIRMED:
    raise RuntimeError(
        "Refusing to run: SCOPE_CONFIRMED is False. Resolve pentest-teva/scope.md's "
        "open items (signed-agreement reference, engagement window, authorized "
        "source IPs, emergency contact) first, then set SCOPE_CONFIRMED = True above."
    )
!python3 teva_pentest_all.py adpaths tevapharm.com


### `ntlminfo` -- passive NTLM auth-challenge domain-name disclosure

Point this at a URL you've already observed returning `401` with `WWW-Authenticate: NTLM` (e.g. from an `adpaths` hit).

In [ ]:
if not SCOPE_CONFIRMED:
    raise RuntimeError(
        "Refusing to run: SCOPE_CONFIRMED is False. Resolve pentest-teva/scope.md's "
        "open items (signed-agreement reference, engagement window, authorized "
        "source IPs, emergency contact) first, then set SCOPE_CONFIRMED = True above."
    )
!python3 teva_pentest_all.py ntlminfo https://REPLACE_ME/observed/401/url


### `smb` / `ldap` -- anonymous internal enumeration

Only meaningful once the engagement grants an internal network vantage point (VPN, jump host, etc.) -- Colab itself has no route to an internal host unless you've separately set that up (not included here).

In [ ]:
if not SCOPE_CONFIRMED:
    raise RuntimeError(
        "Refusing to run: SCOPE_CONFIRMED is False. Resolve pentest-teva/scope.md's "
        "open items (signed-agreement reference, engagement window, authorized "
        "source IPs, emergency contact) first, then set SCOPE_CONFIRMED = True above."
    )
!python3 teva_pentest_all.py smb 10.20.30.40  # replace with a real internal host/IP


In [ ]:
if not SCOPE_CONFIRMED:
    raise RuntimeError(
        "Refusing to run: SCOPE_CONFIRMED is False. Resolve pentest-teva/scope.md's "
        "open items (signed-agreement reference, engagement window, authorized "
        "source IPs, emergency contact) first, then set SCOPE_CONFIRMED = True above."
    )
!python3 teva_pentest_all.py ldap 10.20.30.10  # replace with a real internal host/IP


## 6. Evidence

Anything that actually runs saves raw JSON evidence next to the script, under `evidence/`. Download it from the Colab file browser (left sidebar) before the runtime recycles -- Colab's local disk does not persist across sessions.

In [ ]:
!ls -la evidence/ 2>/dev/null || echo "(no evidence written yet)"
